# Homework #5 | Monitoring

In the module we built all of this by hand - a custom dataclass for the metrics, PostgreSQL for storage, Streamlit and Grafana for dashboards.

In this homework, we will explore an alternative: OpenTelemetry (OTel). This is the industry standard for code instrumentation. Every monitoring framework we mentioned is built on top of it - like Logfire, Langfuse, Arize Phoenix and others.

In this homework we will use OTel directly. We will instrument our RAG with traces, capture metrics as span attributes, persist the spans to SQLite, and build a dashboard from the trace data.

The `starter.py` loads the 72 course lessons, builds a text-search index, and wraps it in a RAGBase instance you can call right away:

In [11]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop, then checks whether the response contains any `function_call` items.

- If there are function calls, it runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is: **no function calls in the latest model response**.


```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Here is what each line does:

- `TracerProvider()` creates the SDK's central configuration object.
  It owns the span processors and decides how spans are built.
- `SimpleSpanProcessor(ConsoleSpanExporter())` wires a processor that
  forwards every finished span to the console exporter, one at a time.
  "Simple" means synchronous and immediate - good for development.
- `trace.set_tracer_provider(provider)` registers the provider
  globally, so every call to `trace.get_tracer(...)` returns a tracer
  backed by it.
- `trace.get_tracer("llm-zoomcamp")` returns a `Tracer` we use to
  create spans. The string is just a label for the instrumentation
  scope - it identifies which part of the code produced the spans.


Put this block at the top of your script, before you import or use
`starter` - so the tracer provider is ready before any code that
might create spans.

With the tracer in hand, you can wrap any block of code in a span:

```python
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

## Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
import importlib
import rag_helper
importlib.reload(rag_helper)
from rag_helper import RAGBase

In [4]:
class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("model", self.model)
            response = super().llm(prompt)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)

In [6]:
traced_rag = RAGTraced(rag.index, rag.llm_client)
answer = traced_rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xaeea4875379a5c945cc754b8b1b596ac",
        "span_id": "0x4603dc84ea7838f5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xba69b0ee14d132a0",
    "start_time": "2026-07-20T16:52:33.911670Z",
    "end_time": "2026-07-20T16:52:33.920431Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "9879e439-83c5-4e61-a7ff-abf149387891",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xaeea4875379a5c945cc754b8b1b596ac",
        

> Total span {} values get: 3

## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

[!NOTE] > The `rag_helper.py` code was rewrote for capturing the `attributes`

In [13]:
import importlib
import rag_helper
importlib.reload(rag_helper)
import opentelemetry
importlib.reload(opentelemetry)

<module 'opentelemetry' (<_frozen_importlib_external.NamespaceLoader object at 0x1583473d0>)>

In [14]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [15]:
class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("model", self.model)
            response = super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)

In [18]:
traced_rag = RAGTraced(rag.index, rag.llm_client)
answer = traced_rag.rag("How does the agentic loop keep calling the model until it stops?")

{
    "name": "search",
    "context": {
        "trace_id": "0xd3d690b7809c08637a0b7c8ce440f10f",
        "span_id": "0xfce95d02a3d7e4e2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6e1857ad7c75d7bd",
    "start_time": "2026-07-20T16:57:43.465123Z",
    "end_time": "2026-07-20T16:57:43.470871Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "9879e439-83c5-4e61-a7ff-abf149387891",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xd3d690b7809c08637a0b7c8ce440f10f",
        

Now re-run the query. How many input tokens do we see?

> 7000 (the closest option.)

## Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

In [19]:
from datetime import datetime
# result from the span
start_time = "2026-07-20T16:57:43.471819Z" 
end_time = "2026-07-20T16:57:45.369973Z"

# Parse ISO 8601 timestamps (replace 'Z' with '+00:00' so fromisoformat can handle it)
start = datetime.fromisoformat(start_time.replace("Z", "+00:00"))
end = datetime.fromisoformat(end_time.replace("Z", "+00:00"))

duration = end - start

print(f"Duration: {duration}")
print(f"Duration (ms): {duration.total_seconds() * 1000:.3f} ms")

Duration: 0:00:01.898154
Duration (ms): 1898.154 ms


## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

<details>
<summary>Let's implement it:</summary>

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```
</details>


Re-run the query from Q1. Which span names appear in the `spans` table?


In [4]:
import importlib
import rag_helper
importlib.reload(rag_helper)
import opentelemetry
importlib.reload(opentelemetry)
import opentelemetry.sdk.trace
importlib.reload(opentelemetry.sdk.trace)
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor


In [8]:
from rag_helper import RAGBase

In [5]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [6]:
provider = TracerProvider()
sqlite_traces = SQLiteSpanExporter("traces.db")

provider.add_span_processor(
    SimpleSpanProcessor(sqlite_traces)
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [9]:
class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("model", self.model)
            response = super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)

In [12]:
traced_rag = RAGTraced(rag.index, rag.llm_client)
answer = traced_rag.rag("How does the agentic loop keep calling the model until it stops?")

In [13]:
import sqlite3

conn = sqlite3.connect("traces.db")
cursor = conn.execute("SELECT * FROM spans")
names = [row[0] for row in cursor.fetchall()]
print(names)

['search', 'llm', 'rag']


In [14]:
import pandas as pd

df = pd.read_sql_query("SELECT rowid, * FROM spans ORDER BY rowid ASC", conn)
df

,rowid,name,start_time,end_time,input_tokens,output_tokens,cost
0,1,search,1784567555718213000,1784567555719787000,NaN,NaN,None
1,2,llm,1784567555720725000,1784567558521719000,7111.0,122.0,None
2,3,rag,1784567555718116000,1784567558523497000,NaN,NaN,None


## Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?


In [15]:
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

df['duration_ms'] = (df['end_time'] - df['start_time']).dt.total_seconds() * 1000
# or in seconds:
df['duration_s'] = (df['end_time'] - df['start_time']).dt.total_seconds()

In [16]:
durations_ms = (
    df[df['name'] != 'rag']
    .groupby('name')['duration_ms']
    .sum()
    .sort_values(ascending=False)
)

durations_ms

name
llm       2800.994
search       1.574
Name: duration_ms, dtype: float64

## Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?


In [17]:
for i in range(1,4):
    print(f"> Iteration #{i}:")
    traced_rag = RAGTraced(rag.index, rag.llm_client)
    traced_rag.rag("How does the agentic loop keep calling the model until it stops?")

> Iteration #1:
> Iteration #2:
> Iteration #3:


In [18]:
df_with_4_iterations = pd.read_sql_query("SELECT rowid, * FROM spans ORDER BY rowid ASC", conn)
df_with_4_iterations

,rowid,name,start_time,end_time,input_tokens,output_tokens,cost
0,1,search,1784567555718213000,1784567555719787000,NaN,NaN,None
1,2,llm,1784567555720725000,1784567558521719000,7111.0,122.0,None
2,3,rag,1784567555718116000,1784567558523497000,NaN,NaN,None
3,4,search,1784567571666347000,1784567571669650000,NaN,NaN,None
4,5,llm,1784567571670684000,1784567573162378000,7111.0,115.0,None
5,6,rag,1784567571666284000,1784567573163592000,NaN,NaN,None
6,7,search,1784567573164444000,1784567573165480000,NaN,NaN,None
7,8,llm,1784567573166115000,1784567574940545000,7111.0,120.0,None
8,9,rag,1784567573164406000,1784567574942984000,NaN,NaN,None
9,10,search,1784567574944598000,1784567574946645000,NaN,NaN,None


In [19]:
llm_df = df_with_4_iterations[df_with_4_iterations['name'] == 'llm']
llm_df['input_tokens'].describe()

count       4.0
mean     7111.0
std         0.0
min      7111.0
25%      7111.0
50%      7111.0
75%      7111.0
max      7111.0
Name: input_tokens, dtype: float64

> They are identical `input_tokens`

In [20]:
llm_df_output_tokens = df_with_4_iterations[df_with_4_iterations['name'] == 'llm']
llm_df_output_tokens['output_tokens'].describe()

count      4.000000
mean     112.500000
std       13.329166
min       93.000000
25%      109.500000
50%      117.500000
75%      120.500000
max      122.000000
Name: output_tokens, dtype: float64